In [1]:
import pandas as pd

df = pd.read_csv("../data/train_transaction.csv")
print(df.shape)

(590540, 394)


In [2]:
missing_pct = df.isnull().mean() * 100  # % missing per column
cols_to_drop = missing_pct[missing_pct > 80].index.tolist()

print(f"Dropping {len(cols_to_drop)} columns:")
print(cols_to_drop)

df = df.drop(columns=cols_to_drop)
print(df.shape) 

Dropping 55 columns:
['dist2', 'D6', 'D7', 'D8', 'D9', 'D12', 'D13', 'D14', 'V138', 'V139', 'V140', 'V141', 'V142', 'V143', 'V144', 'V145', 'V146', 'V147', 'V148', 'V149', 'V150', 'V151', 'V152', 'V153', 'V154', 'V155', 'V156', 'V157', 'V158', 'V159', 'V160', 'V161', 'V162', 'V163', 'V164', 'V165', 'V166', 'V322', 'V323', 'V324', 'V325', 'V326', 'V327', 'V328', 'V329', 'V330', 'V331', 'V332', 'V333', 'V334', 'V335', 'V336', 'V337', 'V338', 'V339']
(590540, 339)


In [3]:
categorical_cols = df.select_dtypes(include='object').columns.tolist()
numeric_cols = df.select_dtypes(include='number').columns.tolist()

print("Categorical columns:", categorical_cols)
print("Number of numeric columns:", len(numeric_cols))

Categorical columns: ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9']
Number of numeric columns: 325


/tmp/ipykernel_164220/12444326.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include='object').columns.tolist()


In [4]:
for col in numeric_cols:
    if col != 'isFraud':  
        df[col] = df[col].fillna(df[col].median())

for col in categorical_cols:
    df[col] = df[col].fillna("missing")

print(df.isnull().sum().sum()) 

0


In [5]:
free_domains = ['gmail.com', 'yahoo.com', 'outlook.com', 'hotmail.com', 'mail.com', 'anonymous.com']

df['is_free_email'] = df['P_emaildomain'].apply(lambda x: 1 if x in free_domains else 0)

print(df['is_free_email'].value_counts())

is_free_email
1    417192
0    173348
Name: count, dtype: int64


/tmp/ipykernel_164220/808650000.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['is_free_email'] = df['P_emaildomain'].apply(lambda x: 1 if x in free_domains else 0)


In [6]:
from sklearn.preprocessing import LabelEncoder

encoders = {}  # save these — we'll need them later for the API to encode new incoming data the same way

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

print(df[categorical_cols].head())

   ProductCD  card4  card6  P_emaildomain  R_emaildomain  M1  M2  M3  M4  M5  \
0          4      1      1             31             31   1   1   1   2   0   
1          4      2      1             16             31   2   2   2   0   1   
2          4      4      2             36             31   1   1   1   0   0   
3          4      2      2             54             31   2   2   2   0   1   
4          1      2      1             16             31   2   2   2   3   2   

   M6  M7  M8  M9  
0   1   2   2   2  
1   1   2   2   2  
2   0   0   0   0  
3   0   2   2   2  
4   2   2   2   2  


In [7]:
df.to_csv("../data/train_cleaned.csv", index=False)
print("Saved cleaned dataset:", df.shape)

Saved cleaned dataset: (590540, 340)


In [8]:
import joblib

joblib.dump(encoders, "../data/label_encoders.pkl")
print("Saved Label Encoders to PKL.")

Saved Label Encoders to PKL.
